# Notebook 48 — Why small statistic differences produce a large collapse

Notebook 47 showed that recalibrating BatchNorm statistics recovers most of the uniform models' loss, yet the saved and
recalibrated statistics differ by only 0.2 to 7% in relative norm. This notebook resolves that, and repairs one flawed
evaluation from Notebook 47.

**1. Shuffled batch statistics.** Notebook 47 evaluated the uniform models with batch statistics on the test partition in
its stored order; that partition is provenance-ordered, so its batches are near-single-class blocks and their statistics
are unrepresentative. The same evaluation is repeated on shuffled test batches.

**2. Per-channel comparison.** For every BatchNorm channel of every uniform model, the ratio of saved to recalibrated
running variance and the standardised mean shift are recorded, and the channels are related to whether their conv.0 filter
is input-connected.

**3. Interpolation sensitivity.** Running mean and variance are moved linearly from the saved values to the recalibrated
values in steps (0, 0.1, ..., 1.0) and macro-F1 is evaluated at each step, with no weight changed.

**4. Which statistic matters.** Recalibrate only the first BatchNorm, only the second, only the means, only the variances.

**Pre-stated criteria.** (a) Shuffled batch statistics on test exceed the saved-statistics macro-F1 by more than 0.10
(then the Notebook 47 batch-statistics number was an ordering artefact). (b) The interpolation curve is not linear: more
than half of the total recovery occurs within the last 30% of the path, or within the first 30% (either shape is a
knife-edge; a linear curve would indicate ordinary sensitivity). (c) At least one single statistic (a layer or a moment)
recovers more than half of the total on its own. CPU or GPU.

In [ ]:
# --- Colab bootstrap ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys, copy, json as _json
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.utils.prune as prune
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from src.config import CFG, PATHS, set_all_seeds
from src.data import load_raw, clean, temporal_within_capture_split
from src import train as TR, models as M, explain as EXP, mitigate
from src.comnet_audit import assign_validation_tiers, calibration_summary, environment_record, write_json
from src.train import load_anchor, predict, per_class_recall_table, feature_columns

assert torch.cuda.is_available(), 'switch to a GPU runtime first'
DEVICE = TR.DEVICE
DATASET = 'ciciot2023'
SEEDS = list(CFG['seeds']); ANCHOR = int(CFG['anchor_seed'])
OUT = PATHS.tables('comnet')
PRACTICAL_LOSS = 0.10
from itertools import combinations
ARCH = 'cnn1d'; KW64 = {'channels': (64, 128)}; KW128 = {'channels': (128, 128)}

KWMLP = {'hidden': (256, 128)}
print('sensitivity analysis ready')

In [ ]:
df = clean(load_raw(DATASET, subsample=True, seed=ANCHOR), DATASET)
splits = temporal_within_capture_split(df, seed=ANCHOR)
feat_cols = feature_columns(df)
print(f'{len(df):,} rows | {df.label.nunique()} classes')

In [ ]:
# Pruning policies on the CNN. Fine-tune loop identical to src.compression.prune_and_finetune.
def prunable(model):
    return [(mod, 'weight') for mod in model.modules() if isinstance(mod, (nn.Linear, nn.Conv1d))]

def layer_names(model):
    return {mod: n for n, mod in model.named_modules()}

def apply_layer_amounts(model, amounts):
    # amounts: {module_name: sparsity}; every prunable layer must be named (no silent defaults)
    m = copy.deepcopy(model); names = layer_names(m)
    for mod, name in prunable(m):
        a = amounts[names[mod]]
        if a > 0: prune.l1_unstructured(mod, name=name, amount=float(a)); prune.remove(mod, name)
    return m

def layer_sparsity(model):
    names = layer_names(model); out = {}
    for mod, name in prunable(model):
        w = getattr(mod, name); out[names[mod]] = float((w == 0).float().mean())
    z = sum(int((getattr(mod, n) == 0).sum()) for mod, n in prunable(model)); n_ = sum(getattr(mod, n).numel() for mod, n in prunable(model))
    out['prunable_sparsity'] = z / n_; out['remaining_nonzero_prunable'] = n_ - z
    return out

def finetune_masked(model, seed, *, ft_epochs=8, batch_size=4096, lr=5e-4, verbose=False):
    set_all_seeds(seed)
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    le = LabelEncoder().fit(df['label'].to_numpy())
    scaler = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
    t = TR.make_tensors(df, splits, feat_cols, le, scaler); Xtr, ytr = t['train']
    model = model.to(DEVICE)
    masks = {(mod, name): (getattr(mod, name) != 0).float() for mod, name in prunable(model)}
    hooks = [getattr(mod, name).register_hook((lambda mk: (lambda g: g * mk))(mk)) for (mod, name), mk in masks.items()]
    w = TR.tempered_class_weights(ytr.numpy(), len(le.classes_))
    crit = nn.CrossEntropyLoss(weight=w); opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for ep in range(ft_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        if verbose: print(f'    ft epoch {ep}')
    for h in hooks: h.remove()
    with torch.no_grad():
        for mod, name in prunable(model): getattr(mod, name).mul_((getattr(mod, name) != 0).float())
    return model.eval(), le, scaler

def save_ckpt(model, le, scaler, path):
    torch.save({'state_dict': model.state_dict(), 'classes': list(le.classes_), 'feat_cols': feat_cols,
                'scaler_mean': scaler.mean_, 'scaler_scale': scaler.scale_}, path)


def load_cell(cell, seed, kw):
    m = M.build(ARCH, len(feat_cols), int(df.label.nunique()), **kw).to(DEVICE)
    m.load_state_dict(torch.load(PATHS.model(DATASET, ARCH, cell, seed), map_location=DEVICE, weights_only=False)['state_dict']); return m.eval()
def loss_table(name, rows, m0f1):
    d = pd.DataFrame(rows); g = d.groupby('cell').test_macro_f1.agg(['mean', 'std', 'min', 'max']); g['loss'] = m0f1 - g['mean']; print(f'\n{name}'); print(g.round(4).to_string()); return d, g
print('helpers ready')
from sklearn.preprocessing import LabelEncoder, StandardScaler
def train_tensors():
    le_ = LabelEncoder().fit(df['label'].to_numpy()); sc = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
    return TR.make_tensors(df, splits, feat_cols, le_, sc)['train'][0]
Xtr_all = train_tensors()
def bn_layers(m): return [mod for mod in m.modules() if isinstance(mod, (nn.BatchNorm1d, nn.BatchNorm2d))]
def recal(model, seed, mode='cumulative', n_batches=200, momentum=0.1):
    m = copy.deepcopy(model)
    for b in bn_layers(m):
        b.reset_running_stats(); b.momentum = None if mode == 'cumulative' else momentum
    m.train(); set_all_seeds(seed); idx = torch.randperm(len(Xtr_all))[: n_batches * 4096]
    with torch.no_grad():
        for i in range(0, len(idx), 4096): m(Xtr_all[idx[i:i + 4096]].to(DEVICE))
    return m.eval()
def predict_batchstats(model, le, scaler, batch_size=4096):
    # train mode (batch statistics) with momentum 0 so running stats are not touched; no dropout in these models.
    # src.train.predict calls model.eval() internally, so the forward pass is done here by hand.
    m = copy.deepcopy(model)
    for b in bn_layers(m): b.momentum = 0.0
    m.train(); sub = df.loc[splits['test']]
    X = torch.tensor(scaler.transform(sub[feat_cols].to_numpy(np.float32)), dtype=torch.float32); outs = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size): outs.append(m(X[i:i + batch_size].to(DEVICE)).cpu())
    assert m.training, 'model left train mode'
    return le.transform(sub['label'].to_numpy()), torch.cat(outs).argmax(1).numpy()
def bn_distance(saved, recalibrated):
    out = []
    for i, (a, b) in enumerate(zip(bn_layers(saved), bn_layers(recalibrated))):
        rm = float((a.running_mean - b.running_mean).norm() / (b.running_mean.norm() + 1e-8)); rv = float((a.running_var - b.running_var).norm() / (b.running_var.norm() + 1e-8))
        out.append({'bn_layer': i, 'rel_mean_distance': rm, 'rel_var_distance': rv, 'saved_var_median': float(a.running_var.median()), 'recal_var_median': float(b.running_var.median())})
    return out
print('helpers ready; training tensor', tuple(Xtr_all.shape))

In [ ]:
# ---------------- 1. Shuffled batch statistics on the test partition ----------------
def predict_batchstats_shuffled(model, le, scaler, seed, batch_size=4096):
    m = copy.deepcopy(model)
    for b in bn_layers(m): b.momentum = 0.0
    m.train(); sub = df.loc[splits['test']]; X = torch.tensor(scaler.transform(sub[feat_cols].to_numpy(np.float32)), dtype=torch.float32); y = le.transform(sub['label'].to_numpy())
    g = torch.Generator().manual_seed(seed); perm = torch.randperm(len(X), generator=g); preds = np.empty(len(X), dtype=np.int64)
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            idx = perm[i:i + batch_size]; preds[idx.numpy()] = m(X[idx].to(DEVICE)).cpu().argmax(1).numpy()
    return y, preds
rows1 = []
for seed in SEEDS:
    m0, le, scaler, _ = load_anchor(DATASET, ARCH, 'M0_paired', seed, arch_kwargs=KW64); mp = load_cell('prune80_paired', seed, KW64)
    yt, p_saved, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test'); y2, p_shuf = predict_batchstats_shuffled(mp, le, scaler, seed)
    rows1.append({'seed': seed, 'saved_running_stats': f1_score(yt, p_saved, average='macro'), 'batch_stats_shuffled_test': f1_score(y2, p_shuf, average='macro')})
    print(f"  seed {seed}: saved {rows1[-1]['saved_running_stats']:.4f} | shuffled batch stats {rows1[-1]['batch_stats_shuffled_test']:.4f}")
d1 = pd.DataFrame(rows1); d1.to_csv(OUT / 'sensitivity_shuffled_batchstats.csv', index=False)
gain_a = float(d1.batch_stats_shuffled_test.mean() - d1.saved_running_stats.mean()); print(f'\n(a) shuffled batch statistics exceed saved by {gain_a:+.3f}:', gain_a > 0.10)

In [ ]:
# ---------------- 2. Per-channel comparison ----------------
rows2 = []
for seed in SEEDS:
    mp = load_cell('prune80_paired', seed, KW64); mc = recal(mp, seed, 'cumulative')
    conv0 = mp.conv[0].weight.detach(); connected = (conv0 != 0).reshape(conv0.shape[0], -1).any(dim=1).cpu().numpy()
    for li, (a, b) in enumerate(zip(bn_layers(mp), bn_layers(mc))):
        va, vb = a.running_var.detach().cpu().numpy(), b.running_var.detach().cpu().numpy(); ma, mb = a.running_mean.detach().cpu().numpy(), b.running_mean.detach().cpu().numpy()
        for ch in range(len(va)):
            rows2.append({'seed': seed, 'bn_layer': li, 'channel': ch, 'connected': bool(connected[ch]) if li == 0 else None, 'saved_var': float(va[ch]), 'recal_var': float(vb[ch]),
                          'var_ratio_saved_over_recal': float((va[ch] + 1e-12) / (vb[ch] + 1e-12)), 'mean_shift_in_recal_sd': float((ma[ch] - mb[ch]) / np.sqrt(vb[ch] + 1e-5))})
d2 = pd.DataFrame(rows2); d2.to_csv(OUT / 'sensitivity_per_channel_stats.csv', index=False)
for li in (0, 1):
    s = d2[d2.bn_layer == li]; print(f'BN layer {li}: |mean shift| in recal sd: median {s.mean_shift_in_recal_sd.abs().median():.3f}, 95th pct {s.mean_shift_in_recal_sd.abs().quantile(0.95):.3f}, max {s.mean_shift_in_recal_sd.abs().max():.3f}')
    print(f'             var ratio saved/recal: median {s.var_ratio_saved_over_recal.median():.3f}, 5th pct {s.var_ratio_saved_over_recal.quantile(0.05):.3f}, 95th pct {s.var_ratio_saved_over_recal.quantile(0.95):.3f}')
s0 = d2[d2.bn_layer == 0]
print('\nBN layer 0 by conv.0 filter connectivity (median |mean shift| / median var ratio):')
print(s0.groupby('connected').agg(n=('channel', 'size'), med_abs_mean_shift=('mean_shift_in_recal_sd', lambda x: x.abs().median()), med_var_ratio=('var_ratio_saved_over_recal', 'median'), saved_var_median=('saved_var', 'median'), recal_var_median=('recal_var', 'median')).round(4).to_string())

In [ ]:
# ---------------- 3. Interpolation from saved to recalibrated statistics ----------------
def interpolate(saved, recalibrated, t):
    m = copy.deepcopy(saved)
    with torch.no_grad():
        for a, b, c in zip(bn_layers(m), bn_layers(saved), bn_layers(recalibrated)):
            a.running_mean.copy_((1 - t) * b.running_mean + t * c.running_mean); a.running_var.copy_((1 - t) * b.running_var + t * c.running_var)
    return m.eval()
steps = [round(x * 0.1, 1) for x in range(11)]; rows3 = []
for seed in SEEDS:
    m0, le, scaler, _ = load_anchor(DATASET, ARCH, 'M0_paired', seed, arch_kwargs=KW64); mp = load_cell('prune80_paired', seed, KW64); mc = recal(mp, seed, 'cumulative')
    for t in steps:
        yt, pp, _ = predict(interpolate(mp, mc, t), df, splits, le, scaler, feat_cols, which='test'); rows3.append({'seed': seed, 't': t, 'macro_f1': f1_score(yt, pp, average='macro')})
d3 = pd.DataFrame(rows3); d3.to_csv(OUT / 'sensitivity_interpolation.csv', index=False)
curve = d3.groupby('t').macro_f1.mean(); print('mean macro-F1 along the path saved (t=0) -> recalibrated (t=1):'); print(curve.round(4).to_string())
total = curve[1.0] - curve[0.0]; first30 = curve[0.3] - curve[0.0]; last30 = curve[1.0] - curve[0.7]
print(f'\n(b) total recovery {total:.3f}; in first 30% of path {first30:.3f} ({first30/total:.0%}); in last 30% {last30:.3f} ({last30/total:.0%})')
knife = bool(first30 / total > 0.5 or last30 / total > 0.5); print('knife-edge (more than half of recovery in one 30% segment):', knife)

In [ ]:
# ---------------- 4. Which statistic matters: one layer or one moment at a time ----------------
def partial_recal(saved, recalibrated, layers=(0, 1), moments=('mean', 'var')):
    m = copy.deepcopy(saved)
    with torch.no_grad():
        for li, (a, c) in enumerate(zip(bn_layers(m), bn_layers(recalibrated))):
            if li not in layers: continue
            if 'mean' in moments: a.running_mean.copy_(c.running_mean)
            if 'var' in moments: a.running_var.copy_(c.running_var)
    return m.eval()
VARIANTS = {'bn0_only': ((0,), ('mean', 'var')), 'bn1_only': ((1,), ('mean', 'var')), 'means_only': ((0, 1), ('mean',)), 'vars_only': ((0, 1), ('var',)), 'all': ((0, 1), ('mean', 'var'))}
rows4 = []
for seed in SEEDS:
    m0, le, scaler, _ = load_anchor(DATASET, ARCH, 'M0_paired', seed, arch_kwargs=KW64); mp = load_cell('prune80_paired', seed, KW64); mc = recal(mp, seed, 'cumulative')
    yt, p0, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test'); base = f1_score(yt, p0, average='macro')
    for name, (layers, moments) in VARIANTS.items():
        yt, pp, _ = predict(partial_recal(mp, mc, layers, moments), df, splits, le, scaler, feat_cols, which='test'); rows4.append({'seed': seed, 'variant': name, 'macro_f1': f1_score(yt, pp, average='macro'), 'gain': f1_score(yt, pp, average='macro') - base})
d4 = pd.DataFrame(rows4); d4.to_csv(OUT / 'sensitivity_partial_recal.csv', index=False)
g4 = d4.groupby('variant').gain.mean(); tot = float(g4['all']); print('mean gain over saved statistics by variant:'); print(g4.round(4).to_string())
single = {k: float(v) for k, v in g4.items() if k != 'all'}; best = max(single, key=single.get)
print(f'\n(c) best single statistic: {best} recovers {single[best]:.3f} of {tot:.3f} ({single[best]/tot:.0%}):', single[best] / tot > 0.5)
verdict = pd.DataFrame([
 {'criterion': 'a_shuffled_batchstats_exceed_saved_by_0.10', 'value': round(gain_a, 4), 'pass': bool(gain_a > 0.10)},
 {'criterion': 'b_interpolation_knife_edge', 'value': f'first30 {first30/total:.0%}, last30 {last30/total:.0%}', 'pass': knife},
 {'criterion': 'c_single_statistic_recovers_gt_half', 'value': f'{best} {single[best]/tot:.0%}', 'pass': bool(single[best] / tot > 0.5)},
]); print(); print(verdict.to_string(index=False)); verdict.to_csv(OUT / 'sensitivity_gate_verdict.csv', index=False)
write_json(OUT / 'sensitivity_environment.json', {'seeds': SEEDS, 'environment': environment_record()})
import subprocess, shutil, glob
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip(); assert _b == 'main', f'checked-out branch is {_b!r}'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True); subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred): shutil.copy(cred, '/root/.git-credentials'); subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
_own = 'notebooks/48_statistic_sensitivity.ipynb'
if os.path.exists(_own):
    d_ = _json.load(open(_own))
    for c in d_.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d_, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/sensitivity_*'), check=True)
r = subprocess.run(['git', 'commit', '-m', 'notebook 48: statistic sensitivity - shuffled batch statistics, per-channel comparison, interpolation path, single-statistic recalibration'], capture_output=True, text=True); print(r.stdout or r.stderr)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed'); print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)